# Browser automation with Selenium — Code-Along Template

**Type into this notebook during class.** The explanation is here; the code is not.

| | |
|---|---|
| **Follow along** | Write the code in each `# TODO` cell. The hint names the functions and arguments. |
| **If you fall behind** | Do not copy frantically. Skip to the next **CHECKPOINT**, run it, rejoin. |
| **If a cell errors** | Later cells depend on it and will fail too. Run the next CHECKPOINT rather than debugging under time pressure. |
| **After class** | The worked version is `D010_C1_5_Selenium_Locators_Guide.ipynb`. Compare, do not copy. |

Cells that are already filled in are deliberate — imports and quick glances at the
data cost class time and teach nothing.

# Selenium Locators — A Complete Guide to Finding Web Elements
## Master XPath, CSS Selectors, and All Selenium Locator Strategies

Before we dive into web scraping, we need to master the most fundamental skill: **locating elements on a web page**. This notebook provides a thorough, hands-on guide to every locator strategy Selenium offers.

### What You'll Learn
- How HTML structure maps to locator strategies
- All 8 Selenium locator types (`By.ID`, `By.NAME`, `By.CLASS_NAME`, `By.TAG_NAME`, `By.LINK_TEXT`, `By.PARTIAL_LINK_TEXT`, `By.CSS_SELECTOR`, `By.XPATH`)
- **CSS Selectors** in depth — combinators, attribute selectors, pseudo-classes
- **XPath** in depth — axes, predicates, text functions
- How to use browser **DevTools** to discover selectors
- A practical mini-scraping project tying it all together

### Prerequisites
- Basic Python knowledge
- A general understanding of HTML tags and attributes
- Chrome browser installed

### Sandbox Websites Used
We use websites that are **explicitly designed** for scraping/testing practice:
- [quotes.toscrape.com](http://quotes.toscrape.com/) — quotes, authors, and tags with clean HTML
- [the-internet.herokuapp.com](https://the-internet.herokuapp.com/) — forms, buttons, tables with clear IDs and names
- [books.toscrape.com](http://books.toscrape.com/) — e-commerce catalogue (used in exercises)

### Estimated Time: ~60 minutes

---
## 1. Setup

Install the required packages (if not already installed) and import libraries.

In [ ]:
# Uncomment and run if packages are not yet installed
# !pip install selenium webdriver-manager pandas

In [6]:
# ── Imports ──────────────────────────────────────────────────────────────────
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time

print("All libraries imported successfully!")

All libraries imported successfully!


In [ ]:
# TODO: compute `chrome_options`, `service`, `driver`
# Hint: Options, chrome_options.add_argument, Service, webdriver.Chrome, driver.set_page_load_timeout
# Name your result: chrome_options, service, driver

---
## 2. Quick HTML / DOM Refresher

Every locator strategy targets some feature of the HTML Document Object Model (DOM). Let's review the key concepts with a small snippet:

```html
<html>
<body>
  <div id="content" class="container">
    <h1>Welcome</h1>
    <form id="loginForm" name="login" class="form-main">
      <label for="user">Username:</label>
      <input id="user" name="username" type="text" class="input-field" />
      <label for="pass">Password:</label>
      <input id="pass" name="password" type="password" class="input-field" />
      <button type="submit" class="btn btn-primary">Log In</button>
    </form>
    <p>Don't have an account? <a href="/register">Sign up here</a></p>
  </div>
</body>
</html>
```

### Key HTML features that locators target

| HTML Feature | Example | Selenium Locator |
|---|---|---|
| `id` attribute | `id="user"` | `By.ID` |
| `name` attribute | `name="username"` | `By.NAME` |
| `class` attribute | `class="input-field"` | `By.CLASS_NAME` |
| Tag name | `<form>`, `<input>`, `<a>` | `By.TAG_NAME` |
| Link text | `Sign up here` | `By.LINK_TEXT` |
| CSS path | `form#loginForm .input-field` | `By.CSS_SELECTOR` |
| XPath expression | `//input[@name='username']` | `By.XPATH` |

### How to see the DOM
In any browser: **Right-click an element → Inspect** (or press `F12`). The Elements panel shows the live DOM tree.

---
## 3. Locator Strategies — Live Examples

We will open sandbox websites in Selenium and practice each locator type.

### 3.1  `By.ID` and `By.NAME`

These are the **simplest and most reliable** locators because `id` values should be unique on a page and `name` values are commonly used on form elements.

We'll demonstrate on the login page at **the-internet.herokuapp.com**.

In [3]:
# ── By.ID and By.NAME ────────────────────────────────────────────────────────
driver = setup_driver(headless=True)

try:
    driver.get("https://the-internet.herokuapp.com/login")
    time.sleep(1)
    print(f"Page title : {driver.title}")
    print(f"Current URL: {driver.current_url}\n")

    # ── By.ID ──
    # The username field has id="username"
    username_field = driver.find_element(By.ID, "username")
    print(f"By.ID  → Found element: <{username_field.tag_name}> "
          f"id='{username_field.get_attribute('id')}' "
          f"type='{username_field.get_attribute('type')}'")

    password_field = driver.find_element(By.ID, "password")
    print(f"By.ID  → Found element: <{password_field.tag_name}> "
          f"id='{password_field.get_attribute('id')}' "
          f"type='{password_field.get_attribute('type')}'")

    # ── By.NAME ──
    # Same fields can also be found by their 'name' attribute
    username_by_name = driver.find_element(By.NAME, "username")
    print(f"\nBy.NAME → Found element: <{username_by_name.tag_name}> "
          f"name='{username_by_name.get_attribute('name')}'")

    password_by_name = driver.find_element(By.NAME, "password")
    print(f"By.NAME → Found element: <{password_by_name.tag_name}> "
          f"name='{password_by_name.get_attribute('name')}'")

    # ── Quick interaction demo ──
    username_field.send_keys("tomsmith")
    password_field.send_keys("SuperSecretPassword!")
    print("\nTyped credentials into the form fields successfully!")

    # ── Click the Login button ──
    # Using CSS_SELECTOR to find the submit button (mixing locator types is fine!)
    login_button = driver.find_element(By.CSS_SELECTOR, "button[type='submit']")
    print(f"\nFound Login button: <{login_button.tag_name}> text='{login_button.text.strip()}'")
    login_button.click()
    time.sleep(2)

    # ── Verify the result ──
    print(f"\nAfter clicking Login:")
    print(f"  Current URL: {driver.current_url}")
    flash_message = driver.find_element(By.ID, "flash")
    print(f"  Flash message: '{flash_message.text.strip()}'")

except Exception as e:
    print(f"Error: {e}")
finally:
    driver.quit()
    print("\nDriver closed.")

Page title : The Internet
Current URL: https://the-internet.herokuapp.com/login

By.ID  → Found element: <input> id='username' type='text'
By.ID  → Found element: <input> id='password' type='password'

By.NAME → Found element: <input> name='username'
By.NAME → Found element: <input> name='password'

Typed credentials into the form fields successfully!
Driver closed.


**Key takeaways:**
- `By.ID` is the fastest locator because browsers index elements by ID.
- `By.NAME` is useful for form inputs where `name` attributes are always present.
- Both return **a single element** (the first match). Use `find_elements` (plural) for multiple matches.
- Use `.send_keys()` to type into input fields and `.click()` to click buttons/links.
- You can **mix locator types** in the same workflow (e.g., `By.ID` for inputs, `By.CSS_SELECTOR` for the button).

In [ ]:
# ═══ CHECKPOINT — "By.CLASS_NAME and By.TAG_NAME" ═══════════════════════════════════════════════
#
# Run this ONLY if you are behind, or a cell above threw an error and the rest
# stopped working. It rebuilds everything this section needs, so you can rejoin
# here instead of being stuck for the rest of the class.
#
# Running it when you are NOT stuck is harmless. If it leaves you stuck anyway,
# that is a bug in the template, not in your code — say so.

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time
def setup_driver(headless=True):
    """
    Configure and return a Chrome WebDriver instance.

    Args:
        headless (bool): If True, runs the browser without a visible window.
    Returns:
        webdriver.Chrome: A configured Chrome driver.
    """
    chrome_options = Options()
    if headless:
        chrome_options.add_argument('--headless=new')
    chrome_options.add_argument('--disable-gpu')
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--window-size=1920,1080')
    chrome_options.add_argument('--disable-dev-shm-usage')
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)
    driver.set_page_load_timeout(30)
    return driver
driver = setup_driver(headless=True)
try:
    driver.get('https://the-internet.herokuapp.com/login')
    time.sleep(1)
    print(f'Page title : {driver.title}')
    print(f'Current URL: {driver.current_url}\n')
    username_field = driver.find_element(By.ID, 'username')
    print(f"By.ID  → Found element: <{username_field.tag_name}> id='{username_field.get_attribute('id')}' type='{username_field.get_attribute('type')}'")
    password_field = driver.find_element(By.ID, 'password')
    print(f"By.ID  → Found element: <{password_field.tag_name}> id='{password_field.get_attribute('id')}' type='{password_field.get_attribute('type')}'")
    username_by_name = driver.find_element(By.NAME, 'username')
    print(f"\nBy.NAME → Found element: <{username_by_name.tag_name}> name='{username_by_name.get_attribute('name')}'")
    password_by_name = driver.find_element(By.NAME, 'password')
    print(f"By.NAME → Found element: <{password_by_name.tag_name}> name='{password_by_name.get_attribute('name')}'")
    username_field.send_keys('tomsmith')
    password_field.send_keys('SuperSecretPassword!')
    print('\nTyped credentials into the form fields successfully!')
    login_button = driver.find_element(By.CSS_SELECTOR, "button[type='submit']")
    print(f"\nFound Login button: <{login_button.tag_name}> text='{login_button.text.strip()}'")
    login_button.click()
    time.sleep(2)
    print(f'\nAfter clicking Login:')
    print(f'  Current URL: {driver.current_url}')
    flash_message = driver.find_element(By.ID, 'flash')
    print(f"  Flash message: '{flash_message.text.strip()}'")
except Exception as e:
    print(f'Error: {e}')
finally:
    driver.quit()
    print('\nDriver closed.')

print("Checkpoint restored. Ready for: By.CLASS_NAME and By.TAG_NAME")

### 3.2  `By.CLASS_NAME` and `By.TAG_NAME`

- **`By.CLASS_NAME`** finds elements that have a given CSS class. Note: you can only pass **one class name** (no spaces).
- **`By.TAG_NAME`** matches by the HTML tag (`div`, `span`, `a`, etc.).

We'll use **quotes.toscrape.com** — a clean site with well-structured classes.

In [ ]:
# TODO: compute `driver`, `quotes`, `authors`
# Hint: setup_driver, driver.get, time.sleep, driver.find_elements, driver.quit
# Name your result: driver, quotes, authors, spans

**Key takeaways:**
- `By.CLASS_NAME` only takes **one** class name. For an element with `class="tag-item"`, use `"tag-item"` — **not** `"tag item"` (that's two classes).
- `By.TAG_NAME` is broad — it often returns many elements. Combine it with `find_element` on a parent to narrow scope.
- `find_elements` (plural) returns a **list**, even if empty — it never throws `NoSuchElementException`.

### 3.3  `By.LINK_TEXT` and `By.PARTIAL_LINK_TEXT`

These locators work **only on `<a>` (anchor) elements** and match by the visible text of the link.

- `By.LINK_TEXT` — must match the **full** visible text exactly.
- `By.PARTIAL_LINK_TEXT` — matches if the link text **contains** the given substring.

In [ ]:
# TODO: compute `driver`, `login_link`, `next_link`
# Hint: setup_driver, driver.get, time.sleep, driver.find_element, driver.find_elements
# Name your result: driver, login_link, next_link, goodreads_links

**Key takeaways:**
- These locators are **only for `<a>` tags**. They won't find `<button>` or `<span>` elements.
- `By.LINK_TEXT` is case-sensitive and must match the full visible text.
- `By.PARTIAL_LINK_TEXT` is handy when link text is dynamic or very long.

---
## 4. CSS Selectors — Deep Dive

`By.CSS_SELECTOR` is one of the **most powerful and commonly used** locators. It uses the same selector syntax as CSS stylesheets.

### CSS Selector Cheat Sheet

| Selector | Syntax | What It Matches |
|---|---|---|
| **ID** | `#myId` | Element with `id="myId"` |
| **Class** | `.myClass` | Elements with `class="myClass"` |
| **Tag** | `div` | All `<div>` elements |
| **Tag + Class** | `span.author` | `<span>` elements with class `author` |
| **Attribute** | `[type='text']` | Elements with `type="text"` |
| **Attr contains** | `[href*='page']` | `href` attribute contains `page` |
| **Attr starts** | `[href^='http']` | `href` starts with `http` |
| **Attr ends** | `[href$='.com']` | `href` ends with `.com` |
| **Descendant** | `div span` | Any `<span>` inside a `<div>` (any depth) |
| **Child** | `div > span` | `<span>` that is a **direct** child of `<div>` |
| **Adjacent sibling** | `h2 + p` | `<p>` immediately after an `<h2>` |
| **General sibling** | `h2 ~ p` | Any `<p>` after an `<h2>` at the same level |
| **First child** | `li:first-child` | First `<li>` among its siblings |
| **Nth child** | `li:nth-child(3)` | Third `<li>` among its siblings |
| **Last child** | `li:last-child` | Last `<li>` among its siblings |
| **Multiple classes** | `.tag-item.active` | Element with both classes |

Let's see these in action on **quotes.toscrape.com**:

In [ ]:
# TODO: compute `driver`, `h1`, `first_quote`
# Hint: setup_driver, driver.get, time.sleep, driver.find_element, driver.find_elements
# Name your result: driver, h1, first_quote, first_author

**Key takeaways:**
- CSS selectors are **concise** and **readable**. They are the go-to locator for most scraping tasks.
- Use **attribute selectors** (`[attr*=val]`) when classes or IDs are unreliable or dynamically generated.
- **Combinators** (`>`, `+`, `~`, space) let you express parent-child and sibling relationships.
- **Pseudo-classes** (`:first-child`, `:nth-child()`) let you pick elements by position.

---
## 5. XPath — Deep Dive

XPath (XML Path Language) is the **most flexible** locator. It can traverse the DOM in any direction — down, up, or sideways — and can match on text content.

### XPath Cheat Sheet

| Expression | What It Does |
|---|---|
| `//tag` | Select all `<tag>` elements anywhere |
| `/html/body/div` | Absolute path from root (fragile!) |
| `//div[@class='quote']` | `<div>` with exact class |
| `//span[text()='Albert Einstein']` | `<span>` with exact text |
| `//span[contains(text(), 'Einstein')]` | `<span>` whose text contains substring |
| `//span[starts-with(@class, 'auth')]` | `<span>` whose class starts with `auth` |
| `//div[@class='quote'][1]` | First quote `<div>` |
| `//div[@class='quote'][last()]` | Last quote `<div>` |
| `//div[@class='quote'][position()<=3]` | First 3 quote `<div>`s |
| `//small[@class='author']/parent::span` | Parent `<span>` of the author element |
| `//span[@class='text']/following-sibling::span` | Next sibling `<span>` after the text |
| `//small[@class='author']/ancestor::div` | Any ancestor `<div>` of the author |
| `//a[contains(@href, 'author')]` | Links whose href contains 'author' |

### Absolute vs Relative XPath
- **Absolute** (`/html/body/div[1]/div[2]`): Starts from the root. Very fragile — breaks if the page structure changes.
- **Relative** (`//div[@class='quote']`): Starts with `//` and searches anywhere. Preferred for reliability.

Let's explore XPath on **quotes.toscrape.com**:

In [ ]:
# TODO: compute `driver`, `quotes`, `first_author`
# Hint: setup_driver, driver.get, time.sleep, driver.find_elements, driver.find_element
# Name your result: driver, quotes, first_author, einstein

**Key takeaways:**
- Always prefer **relative XPath** (`//`) over absolute (`/html/body/...`).
- XPath can match by **text content** — CSS selectors cannot.
- **Axes** (`parent::`, `following-sibling::`, `ancestor::`) let you navigate up and sideways — CSS selectors can only go down.
- Use **predicates** (`[1]`, `[last()]`, `[position()<=3]`) for positional matching.
- XPath is verbose but unmatched in flexibility.

---
## 6. Locator Strategy Comparison

Here's a side-by-side comparison to help you choose the right locator:

| Locator | Syntax Example | Speed | Readability | Flexibility | When to Use |
|---|---|---|---|---|---|
| `By.ID` | `By.ID, "username"` | Fastest | Excellent | Low | Element has a unique `id` |
| `By.NAME` | `By.NAME, "email"` | Fast | Excellent | Low | Form inputs with `name` attr |
| `By.CLASS_NAME` | `By.CLASS_NAME, "author"` | Fast | Good | Low | Single, stable class name |
| `By.TAG_NAME` | `By.TAG_NAME, "table"` | Fast | Good | Low | Few elements of that tag |
| `By.LINK_TEXT` | `By.LINK_TEXT, "Login"` | Fast | Excellent | Low | Exact link text known |
| `By.PARTIAL_LINK_TEXT` | `By.PARTIAL_LINK_TEXT, "Log"` | Fast | Good | Medium | Partial link text |
| `By.CSS_SELECTOR` | `By.CSS_SELECTOR, "div.quote > span"` | Fast | Good | **High** | Complex selections, production scraping |
| `By.XPATH` | `By.XPATH, "//span[contains(text(),'world')]"` | Slower | Lower | **Highest** | Text matching, upward traversal, last resort |

### General Priority (recommended order)
1. `By.ID` — if available, always prefer this
2. `By.NAME` — especially for forms
3. `By.CSS_SELECTOR` — the workhorse for most tasks
4. `By.XPATH` — when you need text matching or upward navigation
5. `By.CLASS_NAME` / `By.TAG_NAME` / `By.LINK_TEXT` — for simple, specific cases

---
## 7. Using Browser DevTools to Find Selectors

You don't have to write selectors from scratch — your browser can help!

### Step-by-Step Guide (Chrome)

1. **Open the page** you want to scrape in Chrome.
2. **Right-click** on the element you're interested in and select **"Inspect"** (or press `Ctrl+Shift+I` / `Cmd+Option+I`).
3. The **Elements** panel opens with the clicked element highlighted in the DOM tree.
4. **Right-click** on the highlighted element in the Elements panel.
5. Choose **Copy** and then:
   - **Copy selector** — gives you a CSS selector
   - **Copy XPath** — gives you an XPath expression
   - **Copy full XPath** — gives the absolute XPath (avoid this; it's fragile)

### Tips for Better Selectors

- **Don't blindly copy** the browser's XPath — it often generates long, fragile absolute paths.
- **Simplify**: After copying, rewrite using classes, IDs, or meaningful attributes.
- **Test in the Console**: In DevTools, press `Ctrl+F` in the Elements panel and type your CSS selector or XPath to see how many elements match.
  - CSS: type the selector directly (e.g., `div.quote span.text`)
  - XPath: prefix with `//` (e.g., `//span[@class='text']`)
- **Use `$$()` in Console tab**: `$$("div.quote")` returns all matching CSS elements; `$x("//div[@class='quote']")` for XPath.

### Example Workflow
```
1. You want to scrape quote text from quotes.toscrape.com
2. Right-click a quote → Inspect
3. See: <span class="text" itemprop="text">"The world as we..."</span>
4. Write CSS: span.text   or   XPath: //span[@class='text']
5. Test in DevTools search (Ctrl+F): shows "10 of 10" — perfect!
```

---
## 8. Mini Scraping Project — Quotes to Scrape

Let's put everything together. We'll scrape the first 10 quotes from **quotes.toscrape.com** into a pandas DataFrame, using a variety of locator strategies.

**Goal**: For each quote, extract:
- The quote text
- The author name
- The author's profile link
- All associated tags

In [ ]:
# TODO: compute `driver`, `quote_elements`, `data`
# Hint: setup_driver, driver.get, time.sleep, driver.find_elements, pd.DataFrame
# Name your result: driver, quote_elements, data, text

In [ ]:
# Display the scraped data
if not df.empty:
    print("\n── Scraped Data Preview ──")
    display(df)
else:
    print("No data to display.")

**What we demonstrated:**
- `By.CSS_SELECTOR` for the quote container, text, tags, and author link
- `By.XPATH` for the author name (using `.//` for scoped search within the parent element)
- Attribute selectors (`[href*='/author/']`) for finding specific links
- Combining `find_element` (single) and `find_elements` (multiple) appropriately
- Scoped search: calling `find_element` on a **parent element** (not the driver) narrows the search to that element's children

---
## 9. Exercises

Practice what you've learned! Implement each exercise following the TODO prompts.

---

### Exercise 1: Identify Elements on a Test Page

**Website:** `https://the-internet.herokuapp.com/challenging_dom`

This page has a table, several buttons (with dynamic IDs), and a canvas element.

**Tasks:**
1. Open the page with Selenium.
2. Use `By.CSS_SELECTOR` to find **all rows** in the HTML table body.
3. Use `By.XPATH` to find the **button** whose text is "qux".
4. Use `By.TAG_NAME` to find the `<canvas>` element and print whether it is displayed.
5. Print the number of table rows and the text of each cell in the **first row**.

In [ ]:
# ── Exercise 1: Identify Elements on a Test Page ─────────────────────────────
# TODO: Create a driver using setup_driver()

# TODO: Navigate to https://the-internet.herokuapp.com/challenging_dom

# TODO: Use By.CSS_SELECTOR to find all <tr> rows inside the table body ("table tbody tr")
#       Print how many rows you found.

# TODO: Use By.XPATH to find the button with text "qux"
#       Hint: //a[text()='qux']  (the buttons are actually <a> tags on this page)
#       Print the tag name and text of the element you found.

# TODO: Use By.TAG_NAME to find the <canvas> element
#       Print whether it is_displayed().

# TODO: Get the first row from step 2, find all <td> cells inside it,
#       and print the text of each cell.

# TODO: Quit the driver


---

### Exercise 2: CSS vs XPath Selector Challenge

**Website:** `http://quotes.toscrape.com/page/2/`

**Tasks:**
1. Open page 2 of quotes.toscrape.com.
2. Using **CSS selectors only**, extract:
   - All quote texts
   - The author of the **last** quote on the page (use `:last-child`)
3. Using **XPath only**, extract:
   - All quotes that mention the word **"life"** (use `contains(text(), ...)`)
   - The **tags** associated with the **first** quote (navigate from the quote div)
4. Print all results clearly.

In [ ]:
# ── Exercise 2: CSS vs XPath Selector Challenge ──────────────────────────────
# TODO: Create a driver using setup_driver()

# TODO: Navigate to http://quotes.toscrape.com/page/2/

# ── CSS Selectors ──
# TODO: Use By.CSS_SELECTOR to find all quote text elements ("span.text")
#       Print each quote text.

# TODO: Use By.CSS_SELECTOR with :last-child to get the author of the last quote
#       Hint: ".quote:last-child small.author"
#       Print the author name.

# ── XPath ──
# TODO: Use By.XPATH to find all <span> elements whose text contains "life"
#       Hint: //span[@class='text'][contains(text(), 'life')]
#       Print each matching quote.

# TODO: Use By.XPATH to get the tags of the first quote
#       Hint: (//div[@class='quote'])[1]//a[@class='tag']
#       Print the list of tags.

# TODO: Quit the driver


---

### Exercise 3: Scrape Books to a DataFrame

**Website:** `http://books.toscrape.com/`

**Tasks:**
1. Open the homepage of books.toscrape.com.
2. For each of the 20 books on the page, extract:
   - **Title** (use CSS: `h3 > a` and get the `title` attribute)
   - **Price** (use XPath: look for the element with class `price_color`)
   - **Rating** (use CSS: the `<p>` element has class `star-rating` plus `One`/`Two`/`Three`/`Four`/`Five`)
   - **Availability** (use XPath: the element with class `instock availability`)
3. Store the data in a pandas DataFrame and display it.
4. Close the driver.

In [ ]:
# ── Exercise 3: Scrape Books to a DataFrame ─────────────────────────────────
# TODO: Create a driver using setup_driver()

# TODO: Navigate to http://books.toscrape.com/

# TODO: Find all book containers (each book is in an <article> tag with class "product_pod")
#       Hint CSS: "article.product_pod"

# TODO: Loop through each book and extract:
#   - title:        CSS "h3 > a" → .get_attribute("title")
#   - price:        XPath ".//p[@class='price_color']" → .text
#   - rating:       CSS "p.star-rating" → .get_attribute("class") → parse the word after "star-rating "
#   - availability: XPath ".//p[contains(@class, 'availability')]" → .text.strip()

# TODO: Create a pandas DataFrame from your list of dictionaries

# TODO: Display the DataFrame

# TODO: Quit the driver


---
## 10. Solutions

Review these **after** you've attempted the exercises.

---

### Solution — Exercise 1

In [ ]:
# ── Solution: Exercise 1 ─────────────────────────────────────────────────────
driver = setup_driver(headless=True)

try:
    # Step 1: Open the page
    driver.get("https://the-internet.herokuapp.com/challenging_dom")
    time.sleep(1)
    print(f"Page: {driver.title}\n")

    # Step 2: Find all table body rows using CSS selector
    rows = driver.find_elements(By.CSS_SELECTOR, "table tbody tr")
    print(f"Table rows found (CSS): {len(rows)}")

    # Step 3: Find the "qux" button using XPath
    qux_button = driver.find_element(By.XPATH, "//a[text()='qux']")
    print(f"\nXPath '//a[text()=\"qux\"]':")
    print(f"  Tag : <{qux_button.tag_name}>")
    print(f"  Text: '{qux_button.text}'")

    # Step 4: Find the <canvas> element using TAG_NAME
    canvas = driver.find_element(By.TAG_NAME, "canvas")
    print(f"\n<canvas> is displayed: {canvas.is_displayed()}")

    # Step 5: Print cells from the first row
    first_row = rows[0]
    cells = first_row.find_elements(By.TAG_NAME, "td")
    print(f"\nFirst row has {len(cells)} cells:")
    for j, cell in enumerate(cells):
        print(f"  Cell {j+1}: '{cell.text}'")

except Exception as e:
    print(f"Error: {e}")
finally:
    driver.quit()
    print("\nDriver closed.")

In [ ]:
# ═══ CHECKPOINT — "Solution — Exercise 2" ═══════════════════════════════════════════════
#
# Run this ONLY if you are behind, or a cell above threw an error and the rest
# stopped working. It rebuilds everything this section needs, so you can rejoin
# here instead of being stuck for the rest of the class.
#
# Running it when you are NOT stuck is harmless. If it leaves you stuck anyway,
# that is a bug in the template, not in your code — say so.

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time
def setup_driver(headless=True):
    """
    Configure and return a Chrome WebDriver instance.

    Args:
        headless (bool): If True, runs the browser without a visible window.
    Returns:
        webdriver.Chrome: A configured Chrome driver.
    """
    chrome_options = Options()
    if headless:
        chrome_options.add_argument('--headless=new')
    chrome_options.add_argument('--disable-gpu')
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--window-size=1920,1080')
    chrome_options.add_argument('--disable-dev-shm-usage')
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)
    driver.set_page_load_timeout(30)
    return driver
driver = setup_driver(headless=True)
try:
    driver.get('https://the-internet.herokuapp.com/login')
    time.sleep(1)
    print(f'Page title : {driver.title}')
    print(f'Current URL: {driver.current_url}\n')
    username_field = driver.find_element(By.ID, 'username')
    print(f"By.ID  → Found element: <{username_field.tag_name}> id='{username_field.get_attribute('id')}' type='{username_field.get_attribute('type')}'")
    password_field = driver.find_element(By.ID, 'password')
    print(f"By.ID  → Found element: <{password_field.tag_name}> id='{password_field.get_attribute('id')}' type='{password_field.get_attribute('type')}'")
    username_by_name = driver.find_element(By.NAME, 'username')
    print(f"\nBy.NAME → Found element: <{username_by_name.tag_name}> name='{username_by_name.get_attribute('name')}'")
    password_by_name = driver.find_element(By.NAME, 'password')
    print(f"By.NAME → Found element: <{password_by_name.tag_name}> name='{password_by_name.get_attribute('name')}'")
    username_field.send_keys('tomsmith')
    password_field.send_keys('SuperSecretPassword!')
    print('\nTyped credentials into the form fields successfully!')
    login_button = driver.find_element(By.CSS_SELECTOR, "button[type='submit']")
    print(f"\nFound Login button: <{login_button.tag_name}> text='{login_button.text.strip()}'")
    login_button.click()
    time.sleep(2)
    print(f'\nAfter clicking Login:')
    print(f'  Current URL: {driver.current_url}')
    flash_message = driver.find_element(By.ID, 'flash')
    print(f"  Flash message: '{flash_message.text.strip()}'")
except Exception as e:
    print(f'Error: {e}')
finally:
    driver.quit()
    print('\nDriver closed.')
driver = setup_driver(headless=True)
try:
    driver.get('http://quotes.toscrape.com/')
    time.sleep(1)
    print(f'Page title: {driver.title}\n')
    quotes = driver.find_elements(By.CLASS_NAME, 'quote')
    print(f"By.CLASS_NAME 'quote' → Found {len(quotes)} quote blocks")
    authors = driver.find_elements(By.CLASS_NAME, 'author')
    print(f"By.CLASS_NAME 'author' → Found {len(authors)} author elements")
    print(f'  First author: {authors[0].text}')
    print(f'  Second author: {authors[1].text}')
    spans = driver.find_elements(By.TAG_NAME, 'span')
    print(f"\nBy.TAG_NAME 'span' → Found {len(spans)} <span> elements")
    links = driver.find_elements(By.TAG_NAME, 'a')
    print(f"By.TAG_NAME 'a'    → Found {len(links)} <a> elements")
    print('\nFirst 5 link texts:')
    for i, link in enumerate(links[:5]):
        print(f"  {i + 1}. '{link.text}' → href={link.get_attribute('href')}")
except Exception as e:
    print(f'Error: {e}')
finally:
    driver.quit()
    print('\nDriver closed.')
driver = setup_driver(headless=True)
try:
    driver.get('http://quotes.toscrape.com/')
    time.sleep(1)
    login_link = driver.find_element(By.LINK_TEXT, 'Login')
    print(f"By.LINK_TEXT 'Login':")
    print(f'  Text : {login_link.text}')
    print(f"  href : {login_link.get_attribute('href')}")
    next_link = driver.find_element(By.PARTIAL_LINK_TEXT, 'Next')
    print(f"\nBy.PARTIAL_LINK_TEXT 'Next':")
    print(f"  Text : '{next_link.text}'")
    print(f"  href : {next_link.get_attribute('href')}")
    goodreads_links = driver.find_elements(By.PARTIAL_LINK_TEXT, 'Goodreads')
    print(f"\nBy.PARTIAL_LINK_TEXT 'Goodreads': found {len(goodreads_links)} link(s)")
    login_partial = driver.find_elements(By.PARTIAL_LINK_TEXT, 'Log')
    print(f"By.PARTIAL_LINK_TEXT 'Log': found {len(login_partial)} link(s)")
    for link in login_partial:
        print(f"  → '{link.text}'")
except Exception as e:
    print(f'Error: {e}')
finally:
    driver.quit()
    print('\nDriver closed.')
driver = setup_driver(headless=True)
try:
    driver.get('http://quotes.toscrape.com/')
    time.sleep(1)
    print('═══ 1. BASIC SELECTORS ═══')
    h1 = driver.find_element(By.CSS_SELECTOR, 'h1')
    print(f"Tag 'h1'           → Text: '{h1.text}'")
    first_quote = driver.find_element(By.CSS_SELECTOR, '.quote')
    print(f"Class '.quote'     → Tag: <{first_quote.tag_name}>")
    first_author = driver.find_element(By.CSS_SELECTOR, 'small.author')
    print(f"'small.author'     → Text: '{first_author.text}'")
    print('\n═══ 2. ATTRIBUTE SELECTORS ═══')
    about_links = driver.find_elements(By.CSS_SELECTOR, "a[href*='/author/']")
    print(f"a[href*='/author/']  → Found {len(about_links)} author links")
    for link in about_links[:3]:
        print(f"  '{link.text}' → {link.get_attribute('href')}")
    http_links = driver.find_elements(By.CSS_SELECTOR, "a[href^='http']")
    print(f"\na[href^='http']    → Found {len(http_links)} external links")
    page_links = driver.find_elements(By.CSS_SELECTOR, "a[href$='/page/2/']")
    print(f"a[href$='/page/2/'] → Found {len(page_links)} link(s) to page 2")
    print('\n═══ 3. COMBINATORS ═══')
    quote_spans = driver.find_elements(By.CSS_SELECTOR, '.quote span')
    print(f"'.quote span' (descendant)     → Found {len(quote_spans)} spans inside quotes")
    quote_direct_spans = driver.find_elements(By.CSS_SELECTOR, '.quote > span')
    print(f"'.quote > span' (direct child) → Found {len(quote_direct_spans)} direct-child spans")
    print('\n═══ 4. PSEUDO-CLASSES ═══')
    first_quote_text = driver.find_element(By.CSS_SELECTOR, '.quote:first-child .text')
    print(f"'.quote:first-child .text'  → '{first_quote_text.text[:60]}...'")
    third_quote = driver.find_element(By.CSS_SELECTOR, '.quote:nth-child(3) .text')
    print(f"'.quote:nth-child(3) .text' → '{third_quote.text[:60]}...'")
    last_quote = driver.find_element(By.CSS_SELECTOR, '.quote:last-child .text')
    print(f"'.quote:last-child .text'   → '{last_quote.text[:60]}...'")
except Exception as e:
    print(f'Error: {e}')
finally:
    driver.quit()
    print('\nDriver closed.')
driver = setup_driver(headless=True)
try:
    driver.get('http://quotes.toscrape.com/')
    time.sleep(1)
    print('═══ 1. BASIC XPATH ═══')
    quotes = driver.find_elements(By.XPATH, "//div[@class='quote']")
    print(f"//div[@class='quote']                  → {len(quotes)} quotes found")
    first_author = driver.find_element(By.XPATH, "//small[@class='author']")
    print(f"//small[@class='author']               → '{first_author.text}'")
    print('\n═══ 2. TEXT MATCHING ═══')
    einstein = driver.find_elements(By.XPATH, "//small[text()='Albert Einstein']")
    print(f"//small[text()='Albert Einstein']      → {len(einstein)} match(es)")
    world_quotes = driver.find_elements(By.XPATH, "//span[contains(text(), 'world')]")
    print(f"//span[contains(text(), 'world')]      → {len(world_quotes)} match(es)")
    for q in world_quotes[:2]:
        print(f"  '{q.text[:70]}...'")
    print('\n═══ 3. POSITIONAL PREDICATES ═══')
    first_q = driver.find_element(By.XPATH, "(//div[@class='quote'])[1]")
    first_text = first_q.find_element(By.XPATH, ".//span[@class='text']").text
    print(f"(//div[@class='quote'])[1]             → '{first_text[:60]}...'")
    last_q = driver.find_element(By.XPATH, "(//div[@class='quote'])[last()]")
    last_text = last_q.find_element(By.XPATH, ".//span[@class='text']").text
    print(f"(//div[@class='quote'])[last()]        → '{last_text[:60]}...'")
    third_q = driver.find_element(By.XPATH, "(//div[@class='quote'])[3]")
    third_text = third_q.find_element(By.XPATH, ".//span[@class='text']").text
    print(f"(//div[@class='quote'])[3]             → '{third_text[:60]}...'")
    print('\n═══ 4. AXES (parent / sibling / ancestor) ═══')
    author_parent = driver.find_element(By.XPATH, "//small[@class='author']/parent::span")
    print(f'author → parent::span                  → Tag: <{author_parent.tag_name}>')
    print(f"  Full text: '{author_parent.text[:80]}'")
    sibling = driver.find_element(By.XPATH, "(//span[@class='text'])[1]/following-sibling::span")
    print(f"\ntext → following-sibling::span         → '{sibling.text[:60]}'")
    ancestor_div = driver.find_element(By.XPATH, "(//small[@class='author'])[1]/ancestor::div[@class='quote']")
    print(f"author → ancestor::div[@class='quote'] → Tag: <{ancestor_div.tag_name}>, class='{ancestor_div.get_attribute('class')}'")
    print('\n═══ 5. COMBINING CONDITIONS ═══')
    author_links = driver.find_elements(By.XPATH, "//a[contains(@href, '/author/') and string-length(text()) > 0]")
    print(f'Author links with text                 → {len(author_links)} found')
    for a in author_links[:3]:
        print(f"  '{a.text}' → {a.get_attribute('href')}")
except Exception as e:
    print(f'Error: {e}')
finally:
    driver.quit()
    print('\nDriver closed.')
driver = setup_driver(headless=True)
try:
    driver.get('http://quotes.toscrape.com/')
    time.sleep(1)
    print(f'Scraping: {driver.current_url}\n')
    quote_elements = driver.find_elements(By.CSS_SELECTOR, 'div.quote')
    print(f'Found {len(quote_elements)} quotes on the page.\n')
    data = []
    for i, quote_el in enumerate(quote_elements, 1):
        text = quote_el.find_element(By.CSS_SELECTOR, 'span.text').text
        author = quote_el.find_element(By.XPATH, ".//small[@class='author']").text
        author_link_el = quote_el.find_element(By.CSS_SELECTOR, "a[href*='/author/']")
        author_link = author_link_el.get_attribute('href')
        tag_elements = quote_el.find_elements(By.CSS_SELECTOR, 'a.tag')
        tags = [tag.text for tag in tag_elements]
        data.append({'quote': text, 'author': author, 'author_link': author_link, 'tags': ', '.join(tags)})
        print(f'Quote {i}: {author} — {text[:50]}...')
    df = pd.DataFrame(data)
    print(f"\n{'=' * 60}")
    print(f'Successfully scraped {len(df)} quotes into a DataFrame!')
    print(f"{'=' * 60}")
except Exception as e:
    print(f'Error: {e}')
    df = pd.DataFrame()
finally:
    driver.quit()
    print('Driver closed.')
driver = setup_driver(headless=True)
try:
    driver.get('https://the-internet.herokuapp.com/challenging_dom')
    time.sleep(1)
    print(f'Page: {driver.title}\n')
    rows = driver.find_elements(By.CSS_SELECTOR, 'table tbody tr')
    print(f'Table rows found (CSS): {len(rows)}')
    qux_button = driver.find_element(By.XPATH, "//a[text()='qux']")
    print(f"""\nXPath '//a[text()="qux"]':""")
    print(f'  Tag : <{qux_button.tag_name}>')
    print(f"  Text: '{qux_button.text}'")
    canvas = driver.find_element(By.TAG_NAME, 'canvas')
    print(f'\n<canvas> is displayed: {canvas.is_displayed()}')
    first_row = rows[0]
    cells = first_row.find_elements(By.TAG_NAME, 'td')
    print(f'\nFirst row has {len(cells)} cells:')
    for j, cell in enumerate(cells):
        print(f"  Cell {j + 1}: '{cell.text}'")
except Exception as e:
    print(f'Error: {e}')
finally:
    driver.quit()
    print('\nDriver closed.')

print("Checkpoint restored. Ready for: Solution — Exercise 2")

### Solution — Exercise 2

In [ ]:
# ── Solution: Exercise 2 ─────────────────────────────────────────────────────
driver = setup_driver(headless=True)

try:
    driver.get("http://quotes.toscrape.com/page/2/")
    time.sleep(1)
    print(f"Scraping: {driver.current_url}\n")

    # ── CSS: All quote texts ──
    print("═══ CSS: All quote texts ═══")
    quote_texts = driver.find_elements(By.CSS_SELECTOR, "span.text")
    for i, q in enumerate(quote_texts, 1):
        print(f"  {i}. {q.text[:70]}...")

    # ── CSS: Author of the last quote ──
    print("\n═══ CSS: Author of the last quote ═══")
    last_author = driver.find_element(By.CSS_SELECTOR, ".quote:last-child small.author")
    print(f"  Last author: {last_author.text}")

    # ── XPath: Quotes containing "life" ──
    print("\n═══ XPath: Quotes containing 'life' ═══")
    life_quotes = driver.find_elements(
        By.XPATH, "//span[@class='text'][contains(text(), 'life')]"
    )
    if life_quotes:
        for q in life_quotes:
            print(f"  → {q.text[:80]}...")
    else:
        print("  No quotes containing 'life' found on this page.")

    # ── XPath: Tags of the first quote ──
    print("\n═══ XPath: Tags of the first quote ═══")
    first_tags = driver.find_elements(
        By.XPATH, "(//div[@class='quote'])[1]//a[@class='tag']"
    )
    tag_list = [t.text for t in first_tags]
    print(f"  Tags: {tag_list}")

except Exception as e:
    print(f"Error: {e}")
finally:
    driver.quit()
    print("\nDriver closed.")

### Solution — Exercise 3

In [ ]:
# ── Solution: Exercise 3 ─────────────────────────────────────────────────────
driver = setup_driver(headless=True)

try:
    driver.get("http://books.toscrape.com/")
    time.sleep(1)
    print(f"Scraping: {driver.current_url}\n")

    # Find all book containers
    books = driver.find_elements(By.CSS_SELECTOR, "article.product_pod")
    print(f"Found {len(books)} books on the page.\n")

    book_data = []

    for i, book in enumerate(books, 1):
        # Title — CSS selector: h3 > a, read the 'title' attribute
        title = book.find_element(By.CSS_SELECTOR, "h3 > a").get_attribute("title")

        # Price — XPath
        price = book.find_element(By.XPATH, ".//p[@class='price_color']").text

        # Rating — CSS: the <p> tag has classes like "star-rating Three"
        rating_class = book.find_element(By.CSS_SELECTOR, "p.star-rating").get_attribute("class")
        # Extract the rating word (e.g., "Three") from "star-rating Three"
        rating = rating_class.replace("star-rating", "").strip()

        # Availability — XPath
        availability = book.find_element(
            By.XPATH, ".//p[contains(@class, 'availability')]"
        ).text.strip()

        book_data.append({
            "title": title,
            "price": price,
            "rating": rating,
            "availability": availability
        })

        if i <= 5:  # Print first 5 for brevity
            print(f"  {i}. {title[:40]:40s} | {price:8s} | {rating:5s} | {availability}")

    if len(books) > 5:
        print(f"  ... ({len(books) - 5} more books)")

    # Create DataFrame
    df_books = pd.DataFrame(book_data)
    print(f"\nDataFrame shape: {df_books.shape}")

except Exception as e:
    print(f"Error: {e}")
    df_books = pd.DataFrame()
finally:
    driver.quit()
    print("Driver closed.")

In [ ]:
# Display the full books DataFrame
if not df_books.empty:
    display(df_books)
else:
    print("No data to display.")

---
## Summary

In this notebook you learned:

1. **All 8 Selenium locator strategies** — when and how to use each one.
2. **CSS selectors** — from basic (`#id`, `.class`) to advanced (combinators, attribute selectors, pseudo-classes).
3. **XPath** — relative paths, text matching, axes, predicates, and combined conditions.
4. **Browser DevTools** — how to inspect elements and discover/test selectors.
5. **Practical scraping** — combining multiple locators to extract structured data into a DataFrame.

### What's Next?

Now that you're confident with locators, move on to:
- **C2: Introduction to Selenium** — full web scraping with waits, interactions, and data storage.
- **C3: Application of BS & Selenium** — combining Selenium navigation with BeautifulSoup parsing.

### Quick Reference

```python
from selenium.webdriver.common.by import By

# ── Locator strategies ──
driver.find_element(By.ID, "username")                          # unique id
driver.find_element(By.NAME, "email")                           # name attribute
driver.find_element(By.CLASS_NAME, "author")                    # single class
driver.find_element(By.TAG_NAME, "table")                       # HTML tag
driver.find_element(By.LINK_TEXT, "Login")                      # exact link text
driver.find_element(By.PARTIAL_LINK_TEXT, "Log")                # partial link text
driver.find_element(By.CSS_SELECTOR, "div.quote > span.text")   # CSS selector
driver.find_element(By.XPATH, "//span[contains(text(),'world')]")  # XPath

# ── Single vs multiple ──
element  = driver.find_element(By.CSS_SELECTOR, ".quote")       # first match (or error)
elements = driver.find_elements(By.CSS_SELECTOR, ".quote")      # list (may be empty)

# ── Scoped search (from a parent element) ──
parent = driver.find_element(By.CSS_SELECTOR, ".quote")
child  = parent.find_element(By.CSS_SELECTOR, "span.text")      # searches within parent
```